# 02 — Detection Training Stages (YOLOv8)

This notebook captures the staged detector strategy used in the project:
1) external pretraining, 2) bootstrap finetuning, 3) optional domain finetuning.

In [ ]:
from pathlib import Path

import torch
from ultralytics import YOLO

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
ROOT = Path('C:/path/to/private/data')
RUNS = ROOT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)

# stage dataset yaml files
STAGE1_YAML = RUNS / 'stage1_dataset.yaml'
STAGE2_YAML = RUNS / 'stage2_dataset.yaml'
STAGE3_YAML = RUNS / 'stage3_truck_dataset.yaml'

STAGE1_YAML.write_text(
f'''path: {str(ROOT).replace('\\', '/')}
train:
  - cleaned_datasets/indian_vehicle/images
  - cleaned_datasets/indian_number_plates/images
val:
  - cleaned_datasets/indian_vehicle/images
names:
  0: license_plate
'''
)

STAGE2_YAML.write_text(
f'''path: {str(ROOT).replace('\\', '/')}
train:
  - Bootstrap_5k/train/images
val:
  - Bootstrap_5k/val/images
names:
  0: license_plate
'''
)

STAGE3_YAML.write_text(
f'''path: {str(ROOT).replace('\\', '/')}
train: Truck Number Plate/train_images/images/train
val: Truck Number Plate/train_images/images/val
names:
  0: license_plate
'''
)

print('Created yaml files under', RUNS)

In [ ]:
# Stage-1: external pretraining
stage1_model = YOLO('yolov8s.pt')

stage1_model.train(
    data=str(STAGE1_YAML),
    epochs=50, imgsz=640, batch=16, workers=4, device=0,
    project=str(RUNS), name='stage1_external_pretraining', exist_ok=True,
    optimizer='auto', amp=True, patience=15,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5, shear=2.0, perspective=0.0005,
    fliplr=0.5, mosaic=1.0, mixup=0.1
)

In [ ]:
# Stage-2: bootstrap finetuning on corrected pseudo-labels
stage2_weights = RUNS / 'stage1_external_pretraining' / 'weights' / 'best.pt'
stage2_model = YOLO(str(stage2_weights))

stage2_model.train(
    data=str(STAGE2_YAML),
    epochs=60, imgsz=640, batch=16, workers=0, device=0,
    project=str(RUNS), name='stage2_fast_stable', exist_ok=True,
    optimizer='AdamW', lr0=3e-4, lrf=0.01, weight_decay=5e-4,
    amp=True, patience=20,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.08, scale=0.40, shear=2.0, perspective=0.0003,
    fliplr=0.5, mosaic=0.5, mixup=0.05
)

In [ ]:
# Stage-3: optional domain adaptation (example: truck subset)
stage3_weights = RUNS / 'stage2_fast_stable' / 'weights' / 'best.pt'
stage3_model = YOLO(str(stage3_weights))

stage3_model.train(
    data=str(STAGE3_YAML),
    epochs=25, imgsz=640, batch=8, workers=0, device=0,
    project=str(RUNS), name='stage3_truck_finetune', exist_ok=True,
    optimizer='AdamW', lr0=1e-4, lrf=0.01, weight_decay=5e-4,
    amp=True, patience=10,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    degrees=5, translate=0.05, scale=0.2
)

## Deliverable
- Stage-wise checkpoints under `runs/`
- Consistent training config across pretraining and finetuning stages